In [ ]:
# The Google Earth Engine module
import ee

# Geopandas allows us to read the vector layer with the list of tiles.
import geopandas

In [ ]:
ee_prj_name = "ee-pb-dev"  # <==== Replace this with your own EE project string
ee.Authenticate()
ee.Initialize(project=ee_prj_name)

In [ ]:
# The region of interest
vec_tiles_file = "roi_5deg_grid.geojson"
tiles_gdf = geopandas.read_file(vec_tiles_file)

# Output directory on Google Drive
out_gdrive_dir = "cci_biomass_v6_data"

In [ ]:

# Iterate through each row (i.e. Tile) and export the data for that tile.
for index, row in tiles_gdf.iterrows():
    tile_name = row['tile_names']
    print(f"Tile: {tile_name}")

    # Get layer bbox: minx, miny, maxx, maxy
    minx, miny, maxx, maxy = row['geometry'].bounds
    gee_bbox_aoi = ee.Geometry.BBox(minx, miny, maxx, maxy)

    for year in [2007, 2010, 2015, 2020, 2022]:
        cci_biomass_img = ee.Image(f"ESA/CCI/Above_Ground_Biomass/V6_0/{year}").clip(gee_bbox_aoi)

        out_file_name = f"{tile_name}_cci_biomass_v6_{year}"

        # Export the Hansen Forest Change image to your Google Drive
        task = ee.batch.Export.image.toDrive(
            image=cci_biomass_img,
            description=out_file_name,
            folder=out_gdrive_dir,
            scale=100,
            region=gee_bbox_aoi,
            maxPixels=1e13,
            fileFormat="GeoTIFF",
            formatOptions={"cloudOptimized": True, "noData": 0.0},
        )
        task.start()
